# 07. Validación y perfiles finales

Dos representaciones, dos preguntas: **cuánto crece** cada artista (intensidad, nivel de log1p centrado, notebooks 04 y 05) y **cómo crece** (forma, tasa de crecimiento centrada, notebook 06). En las dos, la partición elegida es spectral con DTW y k = 3.

Preguntas de este notebook:

1. ¿Los perfiles son estables o dependen de la semilla y de qué artistas entren?
2. ¿Los perfiles son la popularidad disfrazada?
3. ¿Cómo se combinan intensidad y forma?
4. ¿Cómo se describe cada perfil y quién lo representa?

In [ ]:
# PAQUETES

library(data.table)
library(cluster)
library(mclust)
library(ggplot2)
library(ggrepel)
library(patchwork)

# Tipografias de la presentacion (se dibujan con ragg): Berlin Sans FB Demi
# para titulos y DejaVu Sans Mono para el resto (copia en fuentes/)
systemfonts::register_font("LWR Titulo", plain = "C:/Windows/Fonts/BRLNSDB.TTF")
systemfonts::register_font(
  "LWR Mono",
  plain = "fuentes/DejaVuSansMono.ttf",
  bold = "fuentes/DejaVuSansMono-Bold.ttf"
)

# Primera letra en mayúscula para los nombres de los perfiles en las figuras
capitalizar <- function(x) paste0(toupper(substr(x, 1, 1)), substring(x, 2))

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

ARCHIVO_DISTANCIAS_NIVEL <- file.path("chartmetric_pilotos", "distancias_clustering.rds")
ARCHIVO_PARTICIONES_NIVEL <- file.path("chartmetric_pilotos", "particiones_clustering.rds")
ARCHIVO_DISTANCIAS_FORMA <- file.path("chartmetric_pilotos", "distancias_forma_crecimiento.rds")
ARCHIVO_PARTICIONES_FORMA <- file.path("chartmetric_pilotos", "particiones_forma_crecimiento.rds")
FIGURES_DIR <- "figuras"

# Partición elegida en cada representación
K_ELEGIDO <- 3L

# Spectral: misma configuración que los notebooks 05 y 06
VECINO_ESCALA_LOCAL <- 7L
INICIOS_KMEANS <- 25L

# Estabilidad
N_SEMILLAS_KMEANS <- 20L
N_REMUESTREOS <- 50L
PROPORCION_REMUESTREO <- 0.8
N_PANELES_BALANCEADOS <- 10L

SEMILLA_PRIMIGENIA <- 999199L

primos_seis_cifras <- function() {
  limite <- 999999L
  es_primo <- rep(TRUE, limite)
  es_primo[1] <- FALSE
  for (p in 2:floor(sqrt(limite))) {
    if (es_primo[p]) es_primo[seq(p * p, limite, by = p)] <- FALSE
  }
  candidatos <- which(es_primo)
  candidatos[candidatos >= 100000L]
}

PRIMOS_SEIS_CIFRAS <- primos_seis_cifras()

generar_semillas_primas <- function(n, semilla = SEMILLA_PRIMIGENIA) {
  set.seed(semilla)
  orden <- sample(length(PRIMOS_SEIS_CIFRAS))
  PRIMOS_SEIS_CIFRAS[orden[seq_len(n)]]
}

stopifnot(identical(generar_semillas_primas(3), c(742457L, 895957L, 543139L)))

# La primera semilla es la del k-means de las particiones elegidas;
# las siguientes se usan para la estabilidad
SEMILLAS <- generar_semillas_primas(1 + N_SEMILLAS_KMEANS + N_REMUESTREOS + N_PANELES_BALANCEADOS)
SEMILLA_PRINCIPAL <- SEMILLAS[1]
SEMILLAS_KMEANS <- SEMILLAS[1 + seq_len(N_SEMILLAS_KMEANS)]
SEMILLAS_REMUESTREO <- SEMILLAS[1 + N_SEMILLAS_KMEANS + seq_len(N_REMUESTREOS)]
SEMILLAS_BALANCEADO <- SEMILLAS[1 + N_SEMILLAS_KMEANS + N_REMUESTREOS + seq_len(N_PANELES_BALANCEADOS)]

In [ ]:
# ============================================================
# CARGA: matrices DTW, particiones elegidas, artistas y features
# ============================================================

distancias_nivel <- readRDS(ARCHIVO_DISTANCIAS_NIVEL)
distancias_forma <- readRDS(ARCHIVO_DISTANCIAS_FORMA)

MATRICES <- list(intensidad = distancias_nivel$matrices$dtw, forma = distancias_forma$matrices$dtw)
ids_artistas <- rownames(MATRICES$intensidad)

artistas <- copy(distancias_nivel$artistas)[, artist_id := as.character(artist_id)]
artistas <- artistas[match(ids_artistas, artist_id)]

particiones_nivel <- readRDS(ARCHIVO_PARTICIONES_NIVEL)$particiones
particiones_forma <- readRDS(ARCHIVO_PARTICIONES_FORMA)$particiones

asignacion_desde <- function(particiones, algoritmo_elegido) {
  p <- particiones[algoritmo == algoritmo_elegido & k == K_ELEGIDO]
  if ("distancia" %in% names(p)) p <- p[distancia == "dtw"]
  p[match(ids_artistas, artist_id), cluster]
}

PARTICIONES_ELEGIDAS <- list(
  intensidad = asignacion_desde(particiones_nivel, "espectral"),
  forma = asignacion_desde(particiones_forma, "espectral")
)
PARTICIONES_PAM <- list(
  intensidad = asignacion_desde(particiones_nivel, "pam"),
  forma = asignacion_desde(particiones_forma, "pam")
)

archivos_features <- list.files(pattern = "_features_artista[.]csv$")
features <- fread(archivos_features[which.max(file.mtime(archivos_features))])
features[, artist_id := as.character(artist_id)]

# Sanity check: mismos artistas en el mismo orden en todo
stopifnot(
  identical(rownames(MATRICES$forma), ids_artistas),
  !anyNA(unlist(PARTICIONES_ELEGIDAS)), !anyNA(unlist(PARTICIONES_PAM)),
  setequal(features$artist_id, ids_artistas)
)

cat("Artistas:", length(ids_artistas), "\n")
sapply(PARTICIONES_ELEGIDAS, table)

## 2. ¿Los perfiles son estables?

Tres pruebas, todas medidas con el índice de Rand ajustado (ARI: 1 = misma partición, 0 = acuerdo de azar):

- **Semilla:** spectral repetido con 20 semillas del k-means.
- **Remuestreo:** 50 veces, el 80% de los artistas al azar; se recalcula todo (afinidad, autovectores, k-means) y se compara con la partición completa en los artistas que entraron.
- **Algoritmo:** acuerdo entre spectral y PAM.

In [ ]:
# ============================================================
# SPECTRAL (misma implementación que los notebooks 05 y 06)
# ============================================================

agrupar_espectral_matriz <- function(m, k = K_ELEGIDO, semilla = SEMILLA_PRINCIPAL) {
  escala <- apply(m, 1, function(fila) sort(fila)[VECINO_ESCALA_LOCAL + 1])
  afinidad <- exp(-m^2 / outer(escala, escala))
  diag(afinidad) <- 0
  grado <- rowSums(afinidad)
  vectores <- eigen(afinidad / sqrt(outer(grado, grado)), symmetric = TRUE)$vectors[, seq_len(k), drop = FALSE]
  vectores <- vectores / sqrt(rowSums(vectores^2))
  set.seed(semilla)
  kmeans(vectores, centers = k, nstart = INICIOS_KMEANS, iter.max = 100)$cluster
}

# Control: la implementación reproduce las particiones de los notebooks 05 y 06
stopifnot(
  adjustedRandIndex(agrupar_espectral_matriz(MATRICES$intensidad), PARTICIONES_ELEGIDAS$intensidad) > 1 - 1e-9,
  adjustedRandIndex(agrupar_espectral_matriz(MATRICES$forma), PARTICIONES_ELEGIDAS$forma) > 1 - 1e-9
)

In [ ]:
# ============================================================
# ESTABILIDAD: semilla, remuestreo del 80% y algoritmo
# ============================================================

estabilidad <- rbindlist(lapply(names(MATRICES), function(representacion) {
  m <- MATRICES[[representacion]]
  referencia <- PARTICIONES_ELEGIDAS[[representacion]]

  ari_semilla <- vapply(SEMILLAS_KMEANS, function(s)
    adjustedRandIndex(agrupar_espectral_matriz(m, semilla = s), referencia), numeric(1))

  ari_remuestreo <- vapply(SEMILLAS_REMUESTREO, function(s) {
    set.seed(s)
    muestra <- sort(sample(length(ids_artistas), round(PROPORCION_REMUESTREO * length(ids_artistas))))
    adjustedRandIndex(agrupar_espectral_matriz(m[muestra, muestra]), referencia[muestra])
  }, numeric(1))

  data.table(
    representacion = representacion,
    prueba = c("semilla (20)", "remuestreo 80% (50)", "spectral vs PAM"),
    ari_mediano = round(c(median(ari_semilla), median(ari_remuestreo), adjustedRandIndex(referencia, PARTICIONES_PAM[[representacion]])), 3),
    ari_p10 = round(c(quantile(ari_semilla, 0.1), quantile(ari_remuestreo, 0.1), NA), 3)
  )
}))

estabilidad

## 3. ¿Los perfiles son la popularidad disfrazada?

Composición de cada perfil por decil del marco y prueba chi cuadrado. Sensibilidad: 10 paneles balanceados (igual cantidad de artistas por decil, el mínimo disponible), reclusterizados y comparados con la partición completa.

In [ ]:
# ============================================================
# CONTROL POR DECIL DE POPULARIDAD
# ============================================================

MINIMO_POR_DECIL <- min(table(artistas$decil_marco))

control_decil <- rbindlist(lapply(names(PARTICIONES_ELEGIDAS), function(representacion) {
  tabla <- table(cluster = PARTICIONES_ELEGIDAS[[representacion]], decil = artistas$decil_marco)
  prueba <- suppressWarnings(chisq.test(tabla))

  ari_balanceado <- vapply(SEMILLAS_BALANCEADO, function(s) {
    set.seed(s)
    muestra <- sort(unlist(lapply(split(seq_along(ids_artistas), artistas$decil_marco), function(idx) sample(idx, MINIMO_POR_DECIL))))
    adjustedRandIndex(agrupar_espectral_matriz(MATRICES[[representacion]][muestra, muestra]),
                      PARTICIONES_ELEGIDAS[[representacion]][muestra])
  }, numeric(1))

  data.table(
    representacion = representacion,
    chi2_p_valor = signif(prueba$p.value, 2),
    v_de_cramer = round(sqrt(prueba$statistic / (sum(tabla) * (min(dim(tabla)) - 1))), 3),
    panel_balanceado = paste0(MINIMO_POR_DECIL, " por decil"),
    ari_balanceado_mediano = round(median(ari_balanceado), 3)
  )
}))

composicion_decil <- rbindlist(lapply(names(PARTICIONES_ELEGIDAS), function(representacion) {
  data.table(representacion = representacion, cluster = PARTICIONES_ELEGIDAS[[representacion]], decil = artistas$decil_marco)
}))[, .N, by = .(representacion, cluster, decil)][, pct := round(100 * N / sum(N), 1), by = .(representacion, cluster)]

print(control_decil)
cat("\nPorcentaje de cada perfil por decil (1 = más popular):\n")
dcast(composicion_decil, representacion + cluster ~ decil, value.var = "pct")

## 4. Nombres de los perfiles y caracterización

Los nombres se asignan con reglas sobre las features (notebook 03), no a mano:

- **Intensidad:** orden por pendiente anual mediana de las tres audiencias (estable < moderado < fuerte).
- **Forma:** el grupo más grande es "parejo"; de los otros dos, el de mayor salto de suscriptores de YouTube es "salto en YouTube" y el restante, el de menor aceleración, "enfriamiento".

In [ ]:
# ============================================================
# NOMBRES DE LOS PERFILES POR REGLAS SOBRE LAS FEATURES
# ============================================================

AUDIENCIAS <- c("spotify_followers", "instagram_followers", "youtube_channel_subscribers")

features[, `:=`(
  pendiente_audiencias = rowMeans(.SD[, paste0("pendiente_anual__", AUDIENCIAS), with = FALSE]),
  aceleracion_audiencias = rowMeans(.SD[, paste0("aceleracion__", AUDIENCIAS), with = FALSE])
)]
features <- features[match(ids_artistas, artist_id)]
features[, `:=`(cluster_intensidad = PARTICIONES_ELEGIDAS$intensidad, cluster_forma = PARTICIONES_ELEGIDAS$forma)]

orden_intensidad <- features[, .(pendiente = median(pendiente_audiencias)), by = cluster_intensidad][order(pendiente), cluster_intensidad]
nombres_intensidad <- setNames(c("estable", "moderado", "fuerte"), orden_intensidad)

resumen_forma <- features[, .(
  artistas = .N,
  salto_youtube = median(crecimiento_max__youtube_channel_subscribers),
  aceleracion = median(aceleracion_audiencias)
), by = cluster_forma]
cluster_parejo <- resumen_forma[which.max(artistas), cluster_forma]
otros <- resumen_forma[cluster_forma != cluster_parejo]
cluster_salto <- otros[which.max(salto_youtube), cluster_forma]
cluster_enfriamiento <- otros[cluster_forma != cluster_salto, cluster_forma]
nombres_forma <- setNames(c("parejo", "enfriamiento", "salto en YouTube"), c(cluster_parejo, cluster_enfriamiento, cluster_salto))

features[, `:=`(
  perfil_intensidad = factor(nombres_intensidad[as.character(cluster_intensidad)], levels = c("estable", "moderado", "fuerte")),
  perfil_forma = factor(nombres_forma[as.character(cluster_forma)], levels = c("parejo", "enfriamiento", "salto en YouTube"))
)]

# Sanity check: el enfriamiento desacelera más que los parejos
stopifnot(resumen_forma[cluster_forma == cluster_enfriamiento, aceleracion] < resumen_forma[cluster_forma == cluster_parejo, aceleracion])

resumen_forma[, nombre := nombres_forma[as.character(cluster_forma)]][]

In [ ]:
# ============================================================
# CARACTERIZACION: medianas de las features por perfil
# (promedio de las tres audiencias; nivel = tamaño del artista)
# ============================================================

features[, `:=`(
  nivel_audiencias = rowMeans(.SD[, paste0("nivel_log_medio__", AUDIENCIAS), with = FALSE]),
  crecimiento_total_audiencias = rowMeans(.SD[, paste0("crecimiento_total__", AUDIENCIAS), with = FALSE]),
  volatilidad_audiencias = rowMeans(.SD[, paste0("volatilidad_residuo__", AUDIENCIAS), with = FALSE])
)]

caracterizar <- function(columna_perfil) {
  features[
    ,
    .(
      artistas = .N,
      pendiente_anual_pct = round(100 * median(pendiente_audiencias), 1),
      crecimiento_total_pct = round(100 * (exp(median(crecimiento_total_audiencias)) - 1)),
      aceleracion = round(median(aceleracion_audiencias), 4),
      volatilidad = round(median(volatilidad_audiencias), 4),
      nivel_log = round(median(nivel_audiencias), 1),
      decil_mediano = as.numeric(median(decil_marco))
    ),
    keyby = c(perfil = columna_perfil)
  ]
}

cat("Intensidad:\n"); print(caracterizar("perfil_intensidad"))
cat("\nForma:\n"); caracterizar("perfil_forma")

## 5. ¿Cómo se combinan intensidad y forma?

In [ ]:
# ============================================================
# CRUCE INTENSIDAD x FORMA
# ============================================================

cruce <- features[, .N, by = .(perfil_intensidad, perfil_forma)]
cruce[, pct_de_la_intensidad := round(100 * N / sum(N), 1), by = perfil_intensidad]

prueba_cruce <- suppressWarnings(chisq.test(table(features$perfil_intensidad, features$perfil_forma)))
cat("Chi cuadrado intensidad x forma: p =", signif(prueba_cruce$p.value, 2), "\n\n")

print(dcast(cruce, perfil_intensidad ~ perfil_forma, value.var = "N", fill = 0))

# Escala de grises para el porcentaje; los nombres de los perfiles en
# sus colores de línea (los mismos de las figuras de trayectorias)
COLOR_ETIQUETA_INTENSIDAD <- c(estable = "#C9B800", moderado = "#8B6FD6", fuerte = "#1E8FD8")
COLOR_ETIQUETA_FORMA <- c(parejo = "#E0A100", enfriamiento = "#2F55C9", "salto en YouTube" = "#1E9E5A")
niveles_intensidad <- levels(factor(cruce$perfil_intensidad))
niveles_forma <- levels(factor(cruce$perfil_forma))

# Colores por etiqueta: ggplot avisa que el vector en element_text no es
# oficial, pero lo dibuja bien
# Nombre de cada combinación (sección 07 del guion), sobre su celda
NOMBRES_CRUCE <- data.table(
  perfil_intensidad = c("estable", "moderado", "fuerte", "fuerte"),
  perfil_forma = c("parejo", "parejo", "enfriamiento", "parejo"),
  nombre_cruce = c("Consolidados", "Sostenido\nmoderado", "Explosión y\nenfriamiento", "Fuerte y\nsostenido")
)
cruce <- merge(cruce, NOMBRES_CRUCE, by = c("perfil_intensidad", "perfil_forma"), all.x = TRUE)
cruce[, perfil_intensidad := factor(perfil_intensidad, levels = niveles_intensidad)]
cruce[, perfil_forma := factor(perfil_forma, levels = niveles_forma)]

grafico_cruce <- suppressWarnings(ggplot(cruce, aes(x = perfil_forma, y = perfil_intensidad, fill = pct_de_la_intensidad)) +
  geom_tile(colour = "white", linewidth = 2) +
  geom_text(data = cruce[!is.na(nombre_cruce)], aes(label = nombre_cruce, colour = pct_de_la_intensidad > 50),
            size = 7, family = "LWR Titulo", lineheight = 0.9, nudge_y = 0.13) +
  geom_text(aes(label = paste0(N, " (", format(pct_de_la_intensidad, decimal.mark = ",", trim = TRUE), "%)"),
                colour = pct_de_la_intensidad > 50),
            size = 4.5, family = "LWR Mono", fontface = "bold",
            nudge_y = ifelse(is.na(cruce$nombre_cruce), 0, -0.22)) +
  scale_colour_manual(values = c(`TRUE` = "white", `FALSE` = "#111111"), guide = "none") +
  scale_fill_gradientn(colours = c("#F2F2F2", "#9A9A9A", "#111111"), limits = c(0, 100), name = "% de la fila") +
  scale_x_discrete(labels = function(x) sub("Salto en YouTube", "Salto en
YouTube", capitalizar(x))) +
  scale_y_discrete(labels = capitalizar) +
  labs(
    title = "Cuánto crece y cómo crece",
    subtitle = paste0(
      "Artistas por perfil de intensidad (filas) y de forma (columnas).\n",
      "Entre paréntesis, el porcentaje dentro de cada intensidad"
    ),
    x = "Forma del crecimiento", y = "Intensidad del crecimiento"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    axis.title = element_text(colour = "#4A4A4A", size = 12),
    axis.text.x = element_text(family = "LWR Titulo", size = 28, lineheight = 0.9, margin = margin(t = 8), colour = COLOR_ETIQUETA_FORMA[niveles_forma]),
    axis.text.y = element_text(family = "LWR Titulo", size = 28, margin = margin(r = 8), colour = COLOR_ETIQUETA_INTENSIDAD[niveles_intensidad]),
    legend.title = element_text(size = 11, colour = "#4A4A4A"),
    legend.text = element_text(size = 10, colour = "#4A4A4A"),
    panel.grid = element_blank(),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18)
  ))

suppressWarnings(ggsave(file.path(FIGURES_DIR, "perfiles_01_cruce_intensidad_forma.png"), grafico_cruce,
                        width = 12, height = 8, dpi = 300, bg = "white", device = ragg::agg_png))

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "perfiles_01_cruce_intensidad_forma.png"))


## 6. ¿Quién representa a cada perfil?

El artista representativo es el medoide: el de menor distancia DTW total al resto de su perfil. Se listan también los dos más cercanos a él.

In [ ]:
# ============================================================
# ARTISTAS REPRESENTATIVOS (medoide y dos vecinos)
# ============================================================

representativos <- function(representacion, columna_perfil) {
  m <- MATRICES[[representacion]]
  rbindlist(lapply(levels(features[[columna_perfil]]), function(perfil) {
    idx <- which(features[[columna_perfil]] == perfil)
    suma <- rowSums(m[idx, idx, drop = FALSE])
    orden <- idx[order(suma)]
    medoide <- orden[1]
    cercanos <- idx[order(m[medoide, idx])][2:3]
    data.table(
      representacion = representacion,
      perfil = perfil,
      artistas = length(idx),
      medoide = artistas$artist_name[medoide],
      mas_cercanos = paste(artistas$artist_name[cercanos], collapse = ", ")
    )
  }))
}

rbind(representativos("intensidad", "perfil_intensidad"), representativos("forma", "perfil_forma"))

## 7. Metadata de Come Together por perfil

Cruce con la metadata procesada de Come Together (`dataset_7000_53.csv`, mismo `chartmetric_id`): discográfica multinacional, región y género corto. Mismo formato que las fichas de Come Together. La región sobrerrepresentada es la de mayor cociente entre su peso en el perfil y su peso en el panel (1,35x = 35% más que en el panel). Chi cuadrado por variable para saber si las diferencias entre perfiles superan al azar.

In [ ]:
# ============================================================
# METADATA DE COME TOGETHER POR PERFIL
# ============================================================

ARCHIVO_METADATA_CT <- "dataset_7000_53.csv"

metadata_ct <- fread(ARCHIVO_METADATA_CT, select = c("chartmetric_id", "major_record_label", "genre_short", "country_short"))
metadata_ct[, chartmetric_id := as.character(chartmetric_id)]

perfiles_meta <- merge(
  features[, .(artist_id, perfil_intensidad = as.character(perfil_intensidad), perfil_forma = as.character(perfil_forma))],
  metadata_ct, by.x = "artist_id", by.y = "chartmetric_id", all.x = TRUE
)

# Sanity check: metadata para todos los artistas del panel
stopifnot(nrow(perfiles_meta) == length(ids_artistas), !anyNA(perfiles_meta$genre_short), !anyNA(perfiles_meta$country_short))

top_categorias <- function(x, n) {
  t <- sort(table(x), decreasing = TRUE)[seq_len(min(n, length(unique(x))))]
  paste(sprintf("%s %s%%", names(t), format(round(100 * as.numeric(t) / length(x), 1), decimal.mark = ",")), collapse = " · ")
}

ficha_metadata <- function(columna) {
  peso_panel <- prop.table(table(perfiles_meta$country_short))
  perfiles_meta[
    ,
    {
      peso_perfil <- prop.table(table(factor(country_short, levels = names(peso_panel))))
      cociente <- peso_perfil / peso_panel
      region_sobre <- names(which.max(cociente))
      .(
        artistas = .N,
        multinacional_pct = round(100 * mean(major_record_label, na.rm = TRUE), 1),
        region_sobrerrepresentada = sprintf("%s · %sx", region_sobre, format(round(max(cociente), 2), decimal.mark = ",")),
        top_2_regiones = top_categorias(country_short, 2),
        top_3_generos = top_categorias(genre_short, 3)
      )
    },
    by = .(perfil = get(columna))
  ]
}

prueba_metadata <- function(columna) {
  rbindlist(lapply(c("major_record_label", "country_short", "genre_short"), function(v) {
    d <- perfiles_meta[!is.na(get(v))]
    tabla <- table(d[[columna]], d[[v]])
    prueba <- suppressWarnings(chisq.test(tabla))
    data.table(variable = v, chi2_p = signif(prueba$p.value, 2),
               v_de_cramer = round(sqrt(prueba$statistic / (sum(tabla) * (min(dim(tabla)) - 1))), 2))
  }))
}

for (representacion in c("intensidad", "forma")) {
  columna <- paste0("perfil_", representacion)
  cat("\n====", toupper(representacion), "====\n")
  print(ficha_metadata(columna)[order(match(perfil, levels(features[[columna]])))])
  cat("\nChi cuadrado (perfil x variable):\n")
  print(prueba_metadata(columna))
}

cat("\nPanel completo: multinacional", round(100 * mean(perfiles_meta$major_record_label, na.rm = TRUE), 1), "% |",
    top_categorias(perfiles_meta$country_short, 2), "|", top_categorias(perfiles_meta$genre_short, 3), "\n")

## 8. Figuras de los perfiles

Versiones de trabajo para el video, con la paleta de Come Together (la estética final se revisa al elegir las figuras para plotly y manim).

- **Figura 1, trayectorias por perfil:** todos los artistas del perfil en gris, la mediana en color, la banda del 50% central y dos representantes. Muestra el promedio de las tres audiencias (Spotify, Instagram, YouTube) para que sea legible; el clustering de intensidad usó las 5 variables.
- **Figura 2, mapa espectral:** cada artista ubicado con los autovectores del clustering espectral, coloreado por perfil.

In [ ]:
# ============================================================
# CONFIGURACION DE LAS FIGURAS
# Colores de relleno y versiones más oscuras para las líneas.
# ============================================================

# Intensidad: amarillo, lila y celeste (paleta de Come Together).
# Forma: naranja y azul, vecinos del amarillo y el celeste, y verde
# para el salto en YouTube, un color que no se usa en otro lado.
PALETA_RELLENO <- c(amarillo = "#FFF97A", lila = "#C6AEF6", celeste = "#47B3F7",
                    naranja = "#FFD66B", azul = "#6E8EF0", verde = "#7DD8A2")
PALETA_LINEA <- c(amarillo = "#C9B800", lila = "#8B6FD6", celeste = "#1E8FD8",
                  naranja = "#E0A100", azul = "#2F55C9", verde = "#1E9E5A")

COLOR_PERFIL <- list(
  intensidad = c(estable = "amarillo", moderado = "lila", fuerte = "celeste"),
  forma = c(parejo = "naranja", "salto en YouTube" = "verde", enfriamiento = "azul")
)

# Representantes elegidos (archivo LWR_representantes_perfiles.txt)
REPRESENTANTES <- list(
  intensidad = c("3648" = "Ed Sheeran", "2456" = "Abel Pintos", "2762" = "Taylor Swift",
                 "916560" = "Maria Becerra", "171832" = "Miranda!", "4251908" = "Peso Pluma"),
  forma = c("5381" = "Dua Lipa", "748368" = "Nathy Peluso", "865697" = "Luck Ra",
            "105703" = "Kate Bush", "1034" = "Puddle Of Mudd", "470449" = "Playboi Carti")
)

ORDEN_PERFILES <- list(
  intensidad = c("estable", "moderado", "fuerte"),
  forma = c("parejo", "enfriamiento", "salto en YouTube")
)

archivos_panel <- list.files(pattern = "_preprocesado_.+_long[.]csv$")
panel_figuras <- fread(
  archivos_panel[which.max(file.mtime(archivos_panel))],
  select = c("artist_id", "variable", "month", "valor_suavizado", "crecimiento_mensual")
)[variable %in% AUDIENCIAS]
panel_figuras[, `:=`(artist_id = as.character(artist_id), month = as.Date(month))]
panel_figuras <- panel_figuras[artist_id %in% ids_artistas]

In [ ]:
# ============================================================
# SERIES DE LAS FIGURAS (promedio de las tres audiencias)
# - intensidad: crecimiento acumulado desde el primer mes, en
#   veces (exp de la diferencia de log1p suavizado);
# - forma: tasa de crecimiento mensual centrada por serie, en
#   puntos porcentuales, sin los 2 meses de cada borde.
# ============================================================

meses_panel <- sort(unique(panel_figuras$month))

serie_intensidad <- panel_figuras[
  order(month),
  .(month, acumulado_log = valor_suavizado - valor_suavizado[1]),
  by = .(artist_id, variable)
][, .(valor = exp(mean(acumulado_log))), by = .(artist_id, month)]

serie_forma <- panel_figuras[
  month %in% meses_panel[3:(length(meses_panel) - 2)]
][
  ,
  .(month, centrado = crecimiento_mensual - mean(crecimiento_mensual)),
  by = .(artist_id, variable)
][, .(valor = 100 * mean(centrado)), by = .(artist_id, month)]

agregar_perfil <- function(serie, columna) {
  merge(serie, features[, .(artist_id, perfil = as.character(get(columna)))], by = "artist_id")
}

SERIES_FIGURA <- list(
  intensidad = agregar_perfil(serie_intensidad, "perfil_intensidad"),
  forma = agregar_perfil(serie_forma, "perfil_forma")
)

# Sanity check: todos los artistas, un valor por artista y mes
stopifnot(vapply(SERIES_FIGURA, function(s) uniqueN(s$artist_id) == length(ids_artistas) &&
                   !anyDuplicated(s, by = c("artist_id", "month")), logical(1)))

In [ ]:
# ============================================================
# FIGURA 1: TRAYECTORIAS POR PERFIL
# Barra de porcentajes, paneles con trayectorias y fichas.
# ============================================================

ficha_perfil <- function(columna) {
  features[
    ,
    .(
      artistas = .N,
      crec_anual = sprintf("%+.0f%%", 100 * median(pendiente_audiencias)),
      crec_total = {
        x <- exp(median(crecimiento_total_audiencias))
        if (x >= 2) paste0("x ", format(round(x, 1), decimal.mark = ",")) else sprintf("%+.0f%%", 100 * (x - 1))
      },
      audiencia = {
        a <- exp(median(nivel_audiencias))
        if (a >= 1e6) {
          m <- round(a / 1e6, 1)
          paste(format(m, decimal.mark = ","), ifelse(m == 1, "millón", "millones"))
        } else sprintf("%.0f mil", a / 1e3)
      },
      decil = as.numeric(median(decil_marco))
    ),
    by = .(perfil = as.character(get(columna)))
  ]
}

# con_tabla = FALSE: versión para diapositiva (16:9), sin la tabla de
# cifras, que en la presentación va en las fichas de cada perfil
figura_trayectorias <- function(representacion, con_tabla = TRUE) {
  orden <- ORDEN_PERFILES[[representacion]]
  colores <- COLOR_PERFIL[[representacion]][orden]
  datos <- copy(SERIES_FIGURA[[representacion]])
  datos[, perfil := factor(perfil, levels = orden)]
  fichas <- ficha_perfil(paste0("perfil_", representacion))[, perfil := factor(perfil, levels = orden)]
  fichas[, pct := 100 * artistas / sum(artistas)]

  resumen <- datos[, .(mediana = median(valor), q25 = quantile(valor, 0.25), q75 = quantile(valor, 0.75)),
                   by = .(perfil, month)]
  reps <- datos[artist_id %in% names(REPRESENTANTES[[representacion]])]
  reps[, nombre := REPRESENTANTES[[representacion]][artist_id]]
  etiquetas <- reps[, .SD[which.max(month)], by = artist_id]
  # Las etiquetas que se salen de la escala se anclan al borde y muestran su valor
  limites_y <- if (representacion == "intensidad") c(0.7, 40) else c(-4, 4)
  etiquetas[, fuera := valor < limites_y[1] | valor > limites_y[2]]
  etiquetas[, `:=`(
    valor = pmin(pmax(valor, limites_y[1]), limites_y[2]),
    nombre = fifelse(fuera,
                     if (representacion == "intensidad") paste0(nombre, "
(x ", round(valor), ")") else paste0(nombre, " (fuera de escala)"),
                     nombre)
  )]

  es_intensidad <- representacion == "intensidad"

  barra <- ggplot(fichas, aes(x = pct, y = "perfiles", fill = perfil)) +
    geom_col(width = 1, colour = "white", linewidth = 1.2, orientation = "y", position = position_stack(reverse = TRUE)) +
    geom_text(aes(label = ifelse(pct >= 8, sprintf("%s\n%s%% · n = %d", capitalizar(as.character(perfil)), format(round(pct, 1), decimal.mark = ","), artistas),
                                 # tramos muy angostos (menos del 4%) sin etiqueta: no entra
                                 ifelse(pct >= 4, sprintf("%s%%", format(round(pct, 1), decimal.mark = ",")), ""))),
              position = position_stack(vjust = 0.5, reverse = TRUE), size = 3.6, fontface = "bold",
              lineheight = 0.95, colour = "#1D1E2C", family = "LWR Mono") +
    scale_x_continuous(expand = c(0, 0)) +
    scale_fill_manual(values = setNames(PALETA_RELLENO[colores], orden), guide = "none") +
    theme_void()

  paneles <- ggplot(datos, aes(x = month, y = valor)) +
    geom_line(aes(group = artist_id), colour = "grey80", linewidth = 0.2, alpha = 0.35) +
    geom_ribbon(data = resumen, aes(x = month, ymin = q25, ymax = q75, fill = perfil), inherit.aes = FALSE, alpha = 0.55) +
    geom_line(data = resumen, aes(x = month, y = mediana, colour = perfil), inherit.aes = FALSE, linewidth = 1.5) +
    geom_line(data = reps, aes(group = artist_id), colour = "#1D1E2C", linewidth = 0.7, linetype = "22") +
    geom_text_repel(data = etiquetas, aes(label = nombre), size = 3.4, colour = "#1D1E2C", family = "LWR Mono",
                             direction = "y", hjust = 0, nudge_x = 60, segment.colour = "grey60", min.segment.length = 0) +
    facet_wrap(~ perfil, nrow = 1, labeller = as_labeller(capitalizar)) +
    scale_fill_manual(values = setNames(PALETA_RELLENO[colores], orden), guide = "none") +
    scale_colour_manual(values = setNames(PALETA_LINEA[colores], orden), guide = "none") +
    scale_x_date(breaks = as.Date(paste0(2022:2026, "-01-01")), date_labels = "%Y", expand = expansion(mult = c(0.02, 0.55))) +
    labs(x = NULL,
         y = if (es_intensidad) "Crecimiento acumulado desde marzo 2022 (veces)" else "Crecimiento mensual respecto del ritmo propio (p. p.)") +
    theme_minimal(base_size = 12, base_family = "LWR Mono") +
    theme(strip.text = element_text(face = "bold", size = 14, colour = "#111111"),
          axis.title = element_text(colour = "#4A4A4A", size = 12),
          axis.text = element_text(colour = "#6B6B6B", size = 10),
          panel.grid.minor = element_blank(), panel.grid.major = element_line(colour = "#EDEDED"))

  paneles <- if (es_intensidad) {
    paneles + scale_y_log10(breaks = c(1, 1.5, 2, 3, 5, 10, 20, 40), labels = function(x) paste0("x", format(x, decimal.mark = ","))) +
      coord_cartesian(ylim = c(0.7, 40))
  } else {
    paneles + geom_hline(yintercept = 0, colour = "grey50") + coord_cartesian(ylim = c(-4, 4))
  }

  meta <- ficha_metadata(paste0("perfil_", representacion))
  fichas <- merge(fichas, meta[, .(perfil = factor(perfil, levels = orden), multinacional_pct,
                                   region_sobrerrepresentada, top_2_regiones, top_3_generos)], by = "perfil")
  # Tabla de fichas: una columna de rótulos (eje y, alineada con el eje
  # de los paneles) y debajo de cada perfil sólo las cifras. Arriba las
  # medianas del perfil en grande y en su color; abajo la metadata
  partir <- function(x) strsplit(x, " · ", fixed = TRUE)[[1]]
  filas_tabla <- data.table(
    rotulo = c("Crecimiento anual", "En 4 años", "Audiencia típica", "Decil de popularidad",
               "Multinacional", "Sobrerrepresentada", "Regiones", "", "Géneros", "", ""),
    y = c(10.2, 9.3, 8.4, 7.5, 6.2, 5.55, 4.9, 4.4, 3.75, 3.25, 2.75),
    grande = c(rep(TRUE, 4), rep(FALSE, 7))
  )
  valores_tabla <- rbindlist(lapply(seq_len(nrow(fichas)), function(f) {
    fila <- fichas[f]
    data.table(
      perfil = fila$perfil,
      y = filas_tabla$y,
      grande = filas_tabla$grande,
      texto = c(fila$crec_anual, fila$crec_total, fila$audiencia, as.character(fila$decil),
                paste0(format(fila$multinacional_pct, decimal.mark = ","), "%"),
                fila$region_sobrerrepresentada, partir(fila$top_2_regiones), partir(fila$top_3_generos))
    )
  }))
  color_cifra <- setNames(PALETA_LINEA[colores], orden)
  valores_tabla[, `:=`(
    x = as.integer(factor(perfil, levels = orden)) - 0.5,
    color = fifelse(grande, color_cifra[as.character(perfil)], "#111111")
  )]

  tarjetas <- ggplot(valores_tabla, aes(x = x, y = y, label = texto)) +
    annotate("segment", x = 0, xend = length(orden), y = 6.85, yend = 6.85, colour = "#D9D9D9", linewidth = 0.5) +
    geom_text(data = valores_tabla[grande == TRUE], aes(colour = color), family = "LWR Titulo", size = 7.5) +
    geom_text(data = valores_tabla[grande == FALSE], aes(colour = color), family = "LWR Mono", size = 4.2) +
    scale_colour_identity() +
    scale_x_continuous(limits = c(0, length(orden)), expand = c(0, 0)) +
    scale_y_continuous(breaks = filas_tabla[rotulo != "", y], labels = filas_tabla[rotulo != "", rotulo],
                       limits = c(2.4, 10.7), expand = c(0, 0)) +
    labs(x = NULL, y = NULL) +
    theme_void(base_family = "LWR Mono") +
    theme(axis.text.y = element_text(face = "bold", size = 12, colour = "#4A4A4A", hjust = 1, margin = margin(r = 10)))

  titulo <- if (es_intensidad) "Cuánto crece cada artista: tres perfiles de intensidad" else "Cómo crece cada artista: tres perfiles de forma"
  composicion <- if (con_tabla) {
    (barra / paneles / tarjetas) + plot_layout(heights = c(0.08, 1, 0.8))
  } else {
    (barra / paneles) + plot_layout(heights = c(0.07, 1))
  }
  composicion +
    plot_annotation(
      title = titulo,
      subtitle = paste0(
        "Gris: cada artista. Color: mediana del perfil y 50% central. Línea punteada: artistas representativos.\n",
        "Promedio de Spotify, Instagram y YouTube.",
        if (con_tabla) " Fichas: medianas del perfil y metadata de Come Together." else ""
      ),
      theme = theme(
        plot.title = element_text(family = "LWR Titulo", size = 28, colour = "#111111", margin = margin(b = 8)),
        plot.subtitle = element_text(family = "LWR Mono", size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 12)),
        plot.background = element_rect(fill = "white", colour = NA),
        plot.margin = margin(18, 18, 12, 18)
      )
    )
}

for (representacion in c("intensidad", "forma")) {
  figura <- figura_trayectorias(representacion)
  archivo_figura <- file.path(FIGURES_DIR, paste0("perfiles_02_trayectorias_", representacion, ".png"))
  # patchwork avisa que el tema de la anotación no es válido, pero lo aplica
  suppressWarnings(ggsave(archivo_figura, figura, width = 15, height = 13, dpi = 300, bg = "white", device = ragg::agg_png))
  # El dispositivo del notebook no conoce las tipografías registradas: se muestra el PNG
  IRdisplay::display_png(file = archivo_figura)

  # Versión para diapositiva: sin tabla, proporción 16:9
  archivo_diapositiva <- file.path(FIGURES_DIR, paste0("perfiles_02_trayectorias_", representacion, "_sin_tabla.png"))
  suppressWarnings(ggsave(archivo_diapositiva, figura_trayectorias(representacion, con_tabla = FALSE),
                          width = 13.33, height = 7.5, dpi = 300, bg = "white", device = ragg::agg_png))
}

In [ ]:
# ============================================================
# CIFRAS DE CADA PERFIL PARA LAS FICHAS DE UNA COLUMNA
# Medianas y metadata agregada por perfil (sin datos por artista).
# Archivo local, fuera de git; lo lee
# imagenes_estética/fichas/generar_fichas.py
# ============================================================

fichas_columna <- rbindlist(lapply(c("intensidad", "forma"), function(representacion) {
  columna <- paste0("perfil_", representacion)
  medianas <- ficha_perfil(columna)
  meta <- ficha_metadata(columna)
  merged <- merge(medianas, meta[, .(perfil = as.character(perfil), multinacional_pct,
                                     region_sobrerrepresentada, top_2_regiones, top_3_generos)], by = "perfil")
  merged[, `:=`(representacion = representacion, pct = round(100 * artistas / sum(artistas), 1))]
  merged
}))

ARCHIVO_FICHAS_COLUMNA <- file.path("chartmetric_pilotos", "fichas_perfiles.csv")
fwrite(fichas_columna, ARCHIVO_FICHAS_COLUMNA)

# Sanity check: seis perfiles con todas las cifras
stopifnot(nrow(fichas_columna) == 6, !anyNA(fichas_columna))
fichas_columna[, .(representacion, perfil, artistas, pct, crec_anual, crec_total, audiencia, decil)]


In [ ]:
# ============================================================
# FIGURA 2: MAPA ESPECTRAL
# Cada artista ubicado con los autovectores 2 y 3 de la
# afinidad normalizada (los mismos que usa el k-means del
# clustering espectral, normalizados por fila).
# ============================================================

incrustacion_espectral <- function(m, k = K_ELEGIDO) {
  escala <- apply(m, 1, function(fila) sort(fila)[VECINO_ESCALA_LOCAL + 1])
  afinidad <- exp(-m^2 / outer(escala, escala))
  diag(afinidad) <- 0
  grado <- rowSums(afinidad)
  vectores <- eigen(afinidad / sqrt(outer(grado, grado)), symmetric = TRUE)$vectors[, seq_len(k), drop = FALSE]
  vectores / sqrt(rowSums(vectores^2))
}

mapa <- rbindlist(lapply(c("intensidad", "forma"), function(representacion) {
  v <- incrustacion_espectral(MATRICES[[representacion]])
  data.table(
    representacion = ifelse(representacion == "intensidad", "Intensidad: cuánto crece", "Forma: cómo crece"),
    dimension_1 = v[, 2], dimension_2 = v[, 3],
    perfil = as.character(features[[paste0("perfil_", representacion)]]),
    color = PALETA_RELLENO[COLOR_PERFIL[[representacion]][as.character(features[[paste0("perfil_", representacion)]])]]
  )
}))
mapa[, representacion := factor(representacion, levels = c("Intensidad: cuánto crece", "Forma: cómo crece"))]
mapa[, etiqueta := factor(perfil, levels = c(ORDEN_PERFILES$intensidad, ORDEN_PERFILES$forma))]

colores_mapa <- c(
  setNames(PALETA_RELLENO[COLOR_PERFIL$intensidad], names(COLOR_PERFIL$intensidad)),
  setNames(PALETA_RELLENO[COLOR_PERFIL$forma], names(COLOR_PERFIL$forma))
)

figura_mapa <- ggplot(mapa, aes(x = dimension_1, y = dimension_2, fill = etiqueta)) +
  geom_point(shape = 21, size = 2.2, colour = "grey35", stroke = 0.15, alpha = 0.85) +
  facet_wrap(~ representacion, nrow = 1, scales = "free") +
  scale_fill_manual(values = colores_mapa, labels = capitalizar, name = NULL) +
  labs(
    title = "Mapa espectral de los 1252 artistas",
    subtitle = "Cada punto es un artista, ubicado según los autovectores\ndel clustering espectral sobre DTW",
    x = "Dimensión espectral 1", y = "Dimensión espectral 2"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    axis.text = element_blank(), axis.title = element_text(colour = "#4A4A4A", size = 12),
    panel.grid.minor = element_blank(), panel.grid.major = element_line(colour = "#EDEDED"),
    legend.position = "bottom", legend.text = element_text(face = "bold", size = 12, colour = "#111111"),
    strip.text = element_text(face = "bold", size = 13, colour = "#111111", hjust = 0),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18)
  ) +
  guides(fill = guide_legend(override.aes = list(size = 4), nrow = 1))

ggsave(file.path(FIGURES_DIR, "perfiles_03_mapa_espectral.png"), figura_mapa,
       width = 13, height = 7, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "perfiles_03_mapa_espectral.png"))

# Versión para diapositiva: los dos mapas apilados, sin título (va en la
# diapositiva) y con la leyenda en dos filas
figura_mapa_apilado <- figura_mapa +
  facet_wrap(~ representacion, ncol = 1, scales = "free") +
  labs(title = NULL, subtitle = NULL, x = NULL, y = NULL) +
  guides(fill = guide_legend(override.aes = list(size = 4), nrow = 2)) +
  theme(plot.margin = margin(8, 8, 8, 8))
ggsave(file.path(FIGURES_DIR, "perfiles_03_mapa_espectral_apilado.png"), figura_mapa_apilado,
       width = 6.2, height = 7.5, dpi = 300, bg = "white", device = ragg::agg_png)

In [ ]:
# ============================================================
# EXPORTACION: perfil de cada artista (archivo local, fuera de git)
# ============================================================

perfiles_artistas <- features[, .(artist_id, artist_name, cm_artist_rank, decil_marco, perfil_intensidad, perfil_forma)]

ARCHIVO_PERFILES <- "artistas_ST_perfiles_finales.csv"
fwrite(perfiles_artistas, ARCHIVO_PERFILES)

cat("Exportado:", ARCHIVO_PERFILES, "-", nrow(perfiles_artistas), "artistas\n")

## Lectura

- **Estables.** Intensidad: ARI de 1 frente a la semilla y de 0,92 al remuestrear el 80% de los artistas. Forma: 0,91 al remuestrear; algunas semillas del k-means mueven el grupo chico (p10 de 0,76).
- **No son la popularidad disfrazada.** La composición por decil no difiere entre perfiles (chi cuadrado p = 0,21 y 0,75; V de Cramér menor a 0,1) y los paneles balanceados (107 artistas por decil) reproducen las particiones (ARI 0,96 y 0,99).
- **Intensidad y forma no son independientes** (p < 10⁻¹⁰⁰). Casi todos los estables crecen parejo (579 de 584); el 73% de los de crecimiento fuerte está en enfriamiento (123 de 168). El crecimiento fuerte y sostenido es raro: 20 artistas.
- **Representantes (medoides).** Estable: Nicki Minaj (junto a Rihanna). Moderado: KAYTRANADA. Fuerte: Corey Kent (junto a IVE y Luck Ra). Enfriamiento: Anne Wilson (junto a DannyLux y Rema). El grupo de salto en YouTube (26) queda como posible anomalía de la fuente.
- **Algoritmo.** Spectral y PAM acuerdan en intensidad (ARI 0,69) y no en forma (0,25): PAM aísla los saltos de YouTube. Es la razón de elegir spectral.